# End-to-End NDPI Pipeline Demo

This notebook runs the full preprocessing + RF-DETR training + annotation pipeline.
Update the paths in the next cell to match your data layout.

Steps:
1. Read NDPI+NDPA pairs and generate H5 tiles
2. Postprocess tiles (focus stack + rankings)
3. Split into train/val/test
4. Export COCO (single class, filtered boxes)
5. Train RF-DETR
6. Run annotator on a new slide

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# ---- Update these paths ----
repo_root = Path('/home/ats16/dsci435/Smithsonian_fossil_Sp26')
input_ndpi_dir = Path('/path/to/ndpi_with_ndpa')  # contains .ndpi + matching .ndpi.ndpa
annotation_map_csv = Path('/path/to/annotation_categories.csv')

# Output roots
h5_dir = repo_root / 'output' / 'tiles_h5'
splits_dir = repo_root / 'output' / 'splits'
coco_dir = repo_root / 'output' / 'coco_export'
rfdetr_out = repo_root / 'output' / 'rfdetr_run'
annotator_out = repo_root / 'output' / 'annotator'

# New slide to annotate (not part of train/val/test)
annotate_ndpi_path = Path('/path/to/new_slide.ndpi')

# Hyperparameters (edit as needed)
magnification = 40
tile_size = 1024
overlap = 0.0

rfdetr_model = 'base'
rfdetr_epochs = 60
rfdetr_batch_size = 4
rfdetr_grad_accum = 2
rfdetr_lr = 5e-5
rfdetr_lr_scheduler = 'cosine'
rfdetr_lr_min_factor = 0.1
rfdetr_warmup_epochs = 2
rfdetr_weight_decay = 0.01
rfdetr_imgsz = 1008  # must be divisible by 56 for base/small/nano/large
rfdetr_workers = 6
rfdetr_early_stop = 8
rfdetr_drop_path = 0.1
rfdetr_aug = 'custom'

annotator_conf_thresh = 0.5
annotator_nms_iou = 0.5
annotator_magnification = 20
annotator_overlap = 0.10
annotator_compression = 'best_focal_plane'  # or 'focus_stack'

python_exe = sys.executable

# ---- Helpers ----
def run(cmd):
    print(' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONPATH'] = str(repo_root)
    subprocess.run(cmd, cwd=repo_root, env=env, check=True)

# Basic path checks
assert repo_root.is_dir(), f'Repo not found: {repo_root}'
assert input_ndpi_dir.is_dir(), f'NDPI dir not found: {input_ndpi_dir}'
assert annotation_map_csv.is_file(), f'CSV not found: {annotation_map_csv}'

## 1) Generate H5 tiles from NDPI+NDPA

In [ ]:
h5_dir.mkdir(parents=True, exist_ok=True)
run([
    python_exe, '-m', 'src.preprocessing.generate_tiles',
    '--input_dir', str(input_ndpi_dir),
    '--output_dir', str(h5_dir),
    '--annotation_map_path', str(annotation_map_csv),
    '--magnification', str(magnification),
    '--tile_size', str(tile_size),
    '--overlap', str(overlap),
])

## 2) Postprocess tiles (focus stack + rankings)

In [ ]:
run([
    python_exe, '-m', 'src.preprocessing.postprocess_tiles',
    '--input_path', str(h5_dir),
    '--outputs', 'focus_stacked', 'rankings',
])

## 3) Split slides into train/val/test

In [ ]:
splits_dir.mkdir(parents=True, exist_ok=True)
run([
    python_exe, '-m', 'src.preprocessing.split_data',
    '--input_dir', str(h5_dir),
    '--output_dir', str(splits_dir),
    '--seed', '67',
])

## 4) Export focus-stacked COCO (single class, filtered boxes)

In [ ]:
splits_json = splits_dir / 'train_val_test.json'
coco_dir.mkdir(parents=True, exist_ok=True)
run([
    python_exe, '-m', 'src.preprocessing.export_coco',
    '--h5_root', str(h5_dir),
    '--output_dir', str(coco_dir),
    '--mode', 'focus_stack',
    '--splits_json', str(splits_json),
    '--workers', '4',
    '--image_format', 'jpeg',
    '--single_cls',
    '--filter_bboxes',
])

## 5) Train RF-DETR

In [ ]:
rfdetr_out.mkdir(parents=True, exist_ok=True)
run([
    python_exe, '-m', 'src.models.rfdetr.train',
    '--model', rfdetr_model,
    '--coco_dir', str(coco_dir),
    '--epochs', str(rfdetr_epochs),
    '--batch_size', str(rfdetr_batch_size),
    '--grad_accum', str(rfdetr_grad_accum),
    '--lr', str(rfdetr_lr),
    '--lr_scheduler', rfdetr_lr_scheduler,
    '--lr_min_factor', str(rfdetr_lr_min_factor),
    '--warmup_epochs', str(rfdetr_warmup_epochs),
    '--weight_decay', str(rfdetr_weight_decay),
    '--imgsz', str(rfdetr_imgsz),
    '--output_dir', str(rfdetr_out),
    '--workers', str(rfdetr_workers),
    '--early_stopping_patience', str(rfdetr_early_stop),
    '--drop_path', str(rfdetr_drop_path),
    '--aug_config', rfdetr_aug,
])

## 6) Run annotator on a new slide

In [ ]:
# Update this to the best checkpoint produced in the run directory
checkpoint_path = rfdetr_out / 'checkpoint_best_ema.pth'
assert checkpoint_path.is_file(), f'Checkpoint not found: {checkpoint_path}'

annotator_out.mkdir(parents=True, exist_ok=True)
run([
    python_exe, '-m', 'src.annotator',
    '--ndpi_path', str(annotate_ndpi_path),
    '--output_dir', str(annotator_out),
    '--model_name', 'rfdetr',
    '--checkpoint_path', str(checkpoint_path),
    '--overlap', str(annotator_overlap),
    '--magnification', str(annotator_magnification),
    '--compression_method', annotator_compression,
    '--confidence_threshold', str(annotator_conf_thresh),
    '--nms_iou_threshold', str(annotator_nms_iou),
])